# Policy Investigation — does the RL agent rediscover basic strategy, and where doesn't it?

This notebook investigates the trained blackjack Monte Carlo agents, asking not *whether* they
match the proven-optimal **basic strategy**, but **where they disagree, and why**.

The agent learns from nothing but the win/loss reward at the end of each hand. Basic strategy
is the known optimal play, so it is our ground truth. Each training run saved — per state cell
— the agent's action vs basic's, the agent's Q-values, the visit counts, and a category label;
we analyse those saved records (no retraining needed).

Working discipline: re-derive every headline before trusting it; separate real patterns from
base-rate artefacts (within-group checks); distinguish *failed to learn* from *nothing to
learn* from *genuinely different*; and never rank on noisy estimates.

## 1. Load the evidence
We start on the **baseline** run (constant epsilon = 0.1). `cells` is the per-cell policy diff
(one row per state = player total x hard/soft x dealer upcard); `qtable` is the raw learned
Q-values and visit counts per (state, action).

In [ ]:
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

BASELINE = Path("../runs/20260617-032950_seed42_305b302")  # constant eps=0.1
record = json.loads((BASELINE / "record.json").read_text())

cells = pd.DataFrame(record["diff"]["cells"])
qtable = pd.DataFrame(record["qtable"])

print("run:", record["run_id"])
print("eval:", record["eval"], "| diff thresholds:", record["diff"]["min_visits"], record["diff"]["ev_tol"])
print("cells:", len(cells), "| qtable rows:", len(qtable))
cells.head()

## 2. Re-derive the headline before trusting it
Recompute the category counts and agreement from the table, then cross-check against the
summary saved at train time. If they disagree, that mismatch is the first thing to chase.

In [ ]:
print(cells["category"].value_counts())

agree = cells["category"].eq("agree")
unweighted = agree.mean()
weighted = cells.loc[agree, "visits"].sum() / cells["visits"].sum()
print(f"\nagreement: {unweighted:.1%} unweighted, {weighted:.1%} visit-weighted")
print("saved:", record["diff"]["category_counts"],
      f"| {record['diff']['agreement_unweighted']:.3f} / {record['diff']['agreement_weighted']:.3f}")

## 3. Isolate the disagreements, and just look
Filter to `genuine_disagreement` — well-visited cells where the agent confidently chose a
different action than basic, with a meaningful value gap. `ev_gap` = how far apart the agent
thinks the two actions are. Read them before theorising.

In [ ]:
genuine = cells[cells["category"] == "genuine_disagreement"].copy()
genuine["ev_gap"] = (genuine["agent_q"] - genuine["basic_q"]).abs()
genuine = genuine.sort_values("visits", ascending=False)
genuine[["player_value", "is_soft", "dealer_upcard", "visits",
         "agent_action", "basic_action", "agent_q", "basic_q", "ev_gap"]]

## 4. Does the eyeballed pattern survive a base-rate check?
The rows look dominated by soft hands and doubling — but those cells might just be common. So
compare the disagreement *rate within each group* (P(genuine | soft) vs P(genuine | hard),
etc.), not raw counts. A far-higher within-group rate means real structure, not a base-rate
illusion.

In [ ]:
g = cells.copy()
g["involves_double"] = g["agent_action"].eq("double") | g["basic_action"].eq("double")
gd = g[g["category"] == "genuine_disagreement"]

print("genuine disagreements:", len(gd))
print("  soft:           ", int(gd["is_soft"].sum()), "/", len(gd))
print("  involve double: ", int(gd["involves_double"].sum()), "/", len(gd))

print("\nP(genuine_disagreement | group):")
for col in ["is_soft", "involves_double"]:
    rate = g.groupby(col).apply(
        lambda s: (s["category"] == "genuine_disagreement").mean(), include_groups=False
    )
    print(f"\n by {col}:")
    print(rate)

## 5. One pattern, or two mechanisms?
"Concentrates on doubling" is the *what*; the *why* could be (a) the agent **under-explored**
`double` (so its estimate is noisy) or (b) it sampled `double` plenty and genuinely
**over-values** it. Separate them by counting how often each action was actually tried. The
standard error on Q[double] is approx 1.3 / sqrt(double count), so a few-hundred-sample
estimate is noisy on the scale of the gaps above.

In [ ]:
keys = gd[["player_value", "is_soft", "dealer_upcard"]]
sub = qtable.merge(keys, on=["player_value", "is_soft", "dealer_upcard"])
pivot = sub.pivot_table(index=["player_value", "is_soft", "dealer_upcard"],
                        columns="action", values="n", fill_value=0)
pivot["total"] = pivot.sum(axis=1)
pivot["double_share"] = pivot["double"] / pivot["total"]
pivot.sort_values("double_share")

## 6. Visualize the policy — the diagnosed problem
The action heatmap (hard & soft grids): each cell coloured by category, labelled with the
agent's action, or `agent->basic` where they differ. On the baseline you should see the soft
grid light up red in the doubling band (soft 12-18 vs dealer 4-6, mostly `H->D`): the agent
under-doubles soft hands. We define the plot once and reuse it later.

In [ ]:
def plot_policy_heatmap(cells_df, title=""):
    CATS = ["agree", "near_equal_ev", "genuine_disagreement", "under_visited"]
    COLORS = {"agree": "#cdebc5", "near_equal_ev": "#fde08a",
              "genuine_disagreement": "#f3a3a3", "under_visited": "#d9d9d9"}
    cat_to_int = {c: i for i, c in enumerate(CATS)}
    cmap = ListedColormap([COLORS[c] for c in CATS])
    ACT = {"hit": "H", "stand": "S", "double": "D", "split": "P", "surrender": "R"}
    upcards = list(range(2, 12))

    def grid(ax, df, sub):
        totals = sorted(df["player_value"].unique())
        g = np.full((len(totals), len(upcards)), np.nan)
        for _, r in df.iterrows():
            i, j = totals.index(r["player_value"]), upcards.index(r["dealer_upcard"])
            g[i, j] = cat_to_int[r["category"]]
            lab = ACT[r["agent_action"]]
            if r["agent_action"] != r["basic_action"]:
                lab = f"{ACT[r['agent_action']]}\u2192{ACT[r['basic_action']]}"
            ax.text(j, i, lab, ha="center", va="center", fontsize=7)
        ax.imshow(g, cmap=cmap, vmin=0, vmax=len(CATS) - 1, aspect="auto")
        ax.set_xticks(range(len(upcards))); ax.set_xticklabels(upcards)
        ax.set_yticks(range(len(totals))); ax.set_yticklabels(totals)
        ax.set_xlabel("dealer upcard"); ax.set_ylabel("player total"); ax.set_title(sub)

    fig, axes = plt.subplots(1, 2, figsize=(13, 7))
    grid(axes[0], cells_df[~cells_df["is_soft"]], "Hard totals")
    grid(axes[1], cells_df[cells_df["is_soft"]], "Soft totals")
    fig.legend(handles=[Patch(facecolor=COLORS[c], label=c) for c in CATS], loc="lower center", ncol=4)
    fig.suptitle(f"Learned policy vs basic strategy  {title}".strip())
    plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()

plot_policy_heatmap(cells, "- baseline (eps=0.1): the diagnosed problem")

## 7. Controlled experiment — does *more* exploration help?
Compare the baseline (eps=0.1) to a higher-exploration run (eps=0.3) cell by cell: which cells
flip from agree to disagree (worse) and disagree to agree (better)? This is how we test the
under-exploration hypothesis directly.

In [ ]:
RUN_A = Path("../runs/20260617-032950_seed42_305b302")   # eps=0.1
RUN_B = Path("../runs/20260617-055410_seed42_a8e8468")   # eps=0.3
a = pd.DataFrame(json.loads((RUN_A / "record.json").read_text())["diff"]["cells"])
b = pd.DataFrame(json.loads((RUN_B / "record.json").read_text())["diff"]["cells"])

key = ["player_value", "is_soft", "dealer_upcard"]
m = a.merge(b, on=key, suffixes=("_01", "_03"))
print("eps=0.1:", a["category"].value_counts().to_dict())
print("eps=0.3:", b["category"].value_counts().to_dict())

worse = m[(m["category_01"] == "agree") & (m["category_03"] != "agree")]
better = m[(m["category_01"] != "agree") & (m["category_03"] == "agree")]
print(f"\nagree -> disagree (worse) under eps=0.3: {len(worse)}")
print(f"disagree -> agree (better) under eps=0.3: {len(better)}")
print("\nWORSE:\n", worse[key + ["agent_action_03", "basic_action_03"]].to_string(index=False))
print("\nBETTER:\n", better[key + ["agent_action_03", "basic_action_03"]].to_string(index=False))

## 8. Experiment ledger (consistent precision)
Every run is saved with its policy, so we compare variants by re-evaluating each at the **same**
hand count (200k edge estimates are too noisy and can misrank — they did). This loads each
policy and measures it fresh; fidelity (unweighted agreement, genuine count) is exact and read
from the record. Lower edge and higher agreement are better.

In [ ]:
from blackjack_rl.experiment import load_agent
from blackjack_rl.evaluation.metrics import GreedyPolicy, evaluate_policy
from strategies.basic_strategy import BasicStrategy

def run_label(cfg):
    if cfg.get("epsilon_schedule", "constant") == "constant":
        base = f"const eps={cfg['epsilon']}"
    else:
        base = f"{cfg['epsilon_schedule']} {cfg['epsilon_start']}->{cfg['epsilon_end']}"
    if cfg.get("step_size") is not None:
        base += f", a={cfg['step_size']}"
    return base

EVAL_N, EVAL_SEED = 1_000_000, 0   # ~25-30 min; set 200_000 for a quick (noisy) pass
basic = evaluate_policy(BasicStrategy(), n_hands=EVAL_N, seed=EVAL_SEED)
print(f"basic anchor @ {EVAL_N:,}: {basic.edge*100:.3f}% +/- {basic.std_error*100:.3f}")

rows, seen = [], set()
for path in sorted(glob.glob("../runs/*/record.json")):
    r = json.loads(Path(path).read_text())
    label = run_label(r["config"])
    if label in seen:
        continue
    seen.add(label)
    e = evaluate_policy(GreedyPolicy(load_agent(r)), n_hands=EVAL_N, seed=EVAL_SEED)
    d = r["diff"]
    rows.append({
        "label": label,
        "edge_%": round(e.edge * 100, 3),
        "edge_se": round(e.std_error * 100, 3),
        "agree_unwt_%": round(d["agreement_unweighted"] * 100, 1),
        "genuine": d["category_counts"].get("genuine_disagreement", 0),
    })

ledger = pd.DataFrame(rows).sort_values("edge_%").reset_index(drop=True)
ledger

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(ledger["label"], ledger["edge_%"], yerr=ledger["edge_se"], capsize=4,
       color="#9ecae1", edgecolor="#3182bd")
ax.axhline(basic.edge * 100, color="#d62728", linestyle="--", linewidth=1,
           label=f"basic strategy (optimal) \u2248 {basic.edge*100:.2f}%")
ax.set_ylabel("house edge %  (lower = better)")
ax.set_title("Agent house edge by exploration setting (1M-hand eval)")
ax.legend(); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## 9. The chosen policy (decay + alpha) — the residual *relocates*
The best run overall (linear epsilon 0.3->0 with a constant step-size alpha). Compared to the
baseline it fixes several soft doubles, but introduces a few new close-call flips on stiff
hard hands (the constant-alpha noise made visible). Net it is modestly better (12 vs 15
genuine) — the disagreements move around rather than disappear.

In [ ]:
best = json.loads(Path("../runs/20260617-160345_seed42_852e28a/record.json").read_text())
cells_best = pd.DataFrame(best["diff"]["cells"])
plot_policy_heatmap(cells_best, "- decay+alpha (best overall; residual relocates)")

## 10. Convergence — how much training is enough?
The learning curve from the canonical run: policy churn (greedy cells changed since the last
checkpoint) with epsilon overlaid. Churn falling to a low plateau shows the policy has
converged; with constant-alpha it settles to a small non-zero floor rather than exactly zero.
The knee tells us how many episodes were actually needed.

In [ ]:
lc_runs = [p for p in sorted(glob.glob("../runs/*/record.json"))
           if json.loads(Path(p).read_text()).get("learning_curve")]
assert lc_runs, "no run has learning_curve yet — train one with the current code"
rec = json.loads(Path(lc_runs[-1]).read_text())
lc = pd.DataFrame(rec["learning_curve"])

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(lc["episode"], lc["policy_churn"], marker="o", color="#3182bd", label="policy churn")
ax.set_xlabel("training episodes")
ax.set_ylabel("greedy cells changed since last checkpoint")
ax.set_title(f"Convergence - policy churn vs episodes  ({run_label(rec['config'])})")
ax2 = ax.twinx()
ax2.plot(lc["episode"], lc["epsilon"], color="#d62728", linestyle="--", label="epsilon")
ax2.set_ylabel("epsilon")
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [ln.get_label() for ln in lines], loc="upper right")
plt.tight_layout(); plt.show()

## Findings

**The agent rediscovers basic strategy almost entirely.** Across every configuration, coverage
is complete (zero under-visited cells) and the learned greedy policy matches the proven-optimal
table on ~93-95% of cells, at a house edge near ~1% — a near-optimal player learned from
win/loss rewards alone.

**The residual disagreements are concentrated, and explained — not random.** They cluster on
**doubling decisions** (within-group: doubling cells are ~30x more likely to disagree) and a
few near-tie cells. Drilling in, they split into two mechanisms: rare **soft doubles** the agent
*under-explored* (it rarely tried `double`, so its estimate was noisy), and a few well-sampled
cells it genuinely *over-values*.

**The exploration / bias tradeoff (the core finding).** Fixed-epsilon on-policy Monte Carlo
control trades off two things and cannot win both: *low* epsilon keeps the common "stiff" hands'
values clean (best edge) but never samples the rare soft doubles; *high* epsilon samples them
but biases the values (the on-policy continuation is noisy, depressing `Q[hit]`) and breaks the
stiffs. Decaying epsilon is the textbook fix — but **only with a recency-weighted update**: a
1/N sample average can't forget the early high-epsilon returns, so decay *alone* just behaved
like a mid-epsilon run. A constant step-size alpha (recency weighting) was needed.

**decay + alpha is best overall — but modestly, and the residual relocates.** It reaches the
lowest edge (~0.86% at 1M hands) and the fewest genuine disagreements (12), but it fixes some
soft doubles while its own alpha-noise introduces new close-call flips on stiff hands. The
disagreements *move*; they don't vanish.

**The remaining residual is largely irreducible here.** It is not a training-effort failure —
coverage is complete and the learning curve shows convergence by ~2-3M episodes. It is a limit
of the method + representation: the coarse state can't fully separate soft-double contexts
(can-double vs not), near-tie cells need impractically many samples to resolve, and constant-
alpha adds its own jitter. Closing it further needs a **richer state (Stage 3 splits / pair
awareness)** or a different method — not more episodes.

**Meta-lesson worth keeping.** At 200k eval hands the edge estimates were noisy enough to
*invert* the ranking of the two best configs; only the 1M re-evaluations revealed decay+alpha
as best. Don't rank on noisy estimates.